# 🛵 สมุดงานฝึกสอน AI ตรวจจับหมวกนิรภัย - YOLOv8 Nano ความเร็วสูงสุดสำหรับ Raspberry Pi 4
## ⚡ (YOLOv8n Max FPS Edition: NCNN ARM NEON + 320px Direct Training)

สมุดงานนี้ได้รับการปรับแต่งโมเดล **YOLOv8 Nano (`yolov8n.pt`)** เป็นพิเศษเพื่อแก้ปัญหาโมเดลกระตุก/หน่วงบน **Raspberry Pi 4 (CPU ARM Cortex-A72)** โดยเฉพาะ:

### 🎯 จุดเด่นของสูตร YOLOv8n นี้ที่ทำให้รันบน Pi 4 ลื่นที่สุด:
1. **ใช้โมเดลเสถียร YOLOv8 Nano (`yolov8n.pt`):** โครงสร้างมาตรฐาน เสถียรสูง แม่นยำ และเข้ากันได้ 100% กับโค้ดระบบเดิม
2. **ฝึกสอนที่ขนาด 320x320 โดยตรง (`imgsz=320`):** ลดภาระการคำนวณของซีพียูลง **4 เท่า** (จาก 409,600 พิกเซล เหลือเพียง 102,400 พิกเซล) โมเดลเรียนรู้โครงสร้างภาพที่ขนาดนี้โดยตรง ทำให้แม่นยำสูงและรันได้เร็วสุดขีด
3. **แปลงเป็น NCNN (Tencent NCNN):** คอมไพล์ด้วยชุดคำสั่ง **ARM NEON Assembly** รันบน CPU ของ Raspberry Pi 4 ได้โดยตรงโดยไม่ต้องพึ่งพา PyTorch หนักๆ
4. **แปลงเป็น TFLite INT8 Quantization:** แปลงค่าน้ำหนักเป็นเลขจำนวนเต็ม 8 บิต ขนาดไฟล์ลดลงเหลือเพียง **~3 MB**
5. **คาดการณ์ความเร็วบน Raspberry Pi 4:** **25 - 35+ FPS ลื่นไหลแบบ Real-time!**

---
### 📌 วิธีใช้งาน:
1. ตรวจสอบว่าเปิดใช้ GPU แล้ว (เมนู `Runtime` -> `Change runtime type` -> เลือก `T4 GPU`)
2. กดปุ่ม Play (▶️) ทีละเซลล์ตามลำดับจนจบ ระบบจะเซฟโมเดลทั้งหมดลง Google Drive ให้อัตโนมัติ

### 1️⃣ ตรวจสอบการเปิดใช้งาน GPU บน Google Colab

In [ ]:
# ตรวจสอบว่า Colab เปิดใช้งาน GPU T4 เรียบร้อย
!nvidia-smi

### 2️⃣ เชื่อมต่อ Google Drive และค้นหาไฟล์ `lastdata.zip` อัตโนมัติ
> 💡 **คำแนะนำ:** หากไฟล์ `lastdata.zip` หรือโฟลเดอร์อยู่ในแถบ *"แชร์กับฉัน (Shared with me)"* ให้คลิกขวาที่โฟลเดอร์ -> เลือก *"จัดระเบียบ (Organize)"* -> *"เพิ่มทางลัด (Add shortcut)"* -> เลือก *"ไดรฟ์ของฉัน (My Drive)"*

In [ ]:
import os
import shutil
import zipfile
from google.colab import drive

# 1. เชื่อมต่อ Google Drive
drive.mount('/content/drive')

# 2. ค้นหาตำแหน่งไฟล์ lastdata.zip โดยอัตโนมัติจากทุกโฟลเดอร์หลัก
candidate_paths = [
    '/content/lastdata.zip',
    '/content/drive/MyDrive/lastdata.zip',
    '/content/drive/MyDrive/กันล็อคเด้ออ้าย/lastdata.zip',
    '/content/drive/Shareddrives/กันล็อคเด้ออ้าย/lastdata.zip'
]

zip_file_path = None
for p in candidate_paths:
    if os.path.exists(p):
        zip_file_path = p
        break

# หากไม่พบในจุดหลัก ให้สแกนหาในไดรฟ์อย่างละเอียด
if not zip_file_path:
    print("🔍 กำลังค้นหาไฟล์ lastdata.zip ภายใน Google Drive...")
    for root, dirs, files in os.walk('/content/drive/MyDrive'):
        if 'lastdata.zip' in files:
            zip_file_path = os.path.join(root, 'lastdata.zip')
            break

if zip_file_path and os.path.exists(zip_file_path):
    file_size_mb = os.path.getsize(zip_file_path) / (1024 * 1024)
    print("✅ พบไฟล์ชุดข้อมูลสำเร็จ!")
    print(f"📍 ที่อยู่ไฟล์: {zip_file_path}")
    print(f"📦 ขนาดไฟล์: {file_size_mb:.2f} MB")
else:
    raise FileNotFoundError("❌ ไม่พบไฟล์ lastdata.zip! กรุณาตรวจสอบว่ามีไฟล์นี้ใน Google Drive หรือคลิกขวาเพิ่มทางลัดมาไว้ที่ My Drive")

### 3️⃣ แตกไฟล์ Dataset และตั้งค่า `data.yaml` สำหรับขนาด 320px

In [ ]:
import yaml

DATASET_DIR = '/content/dataset'
if os.path.exists(DATASET_DIR):
    shutil.rmtree(DATASET_DIR)
os.makedirs(DATASET_DIR, exist_ok=True)

print(f"⏳ กำลังแตกไฟล์ {os.path.basename(zip_file_path)} ไปยัง {DATASET_DIR}...")
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(DATASET_DIR)
print("✅ แตกไฟล์ชุดข้อมูลเรียบร้อย!")

# ตรวจสอบโครงสร้างโฟลเดอร์
train_img_dir = os.path.join(DATASET_DIR, 'train', 'images')
val_img_dir = os.path.join(DATASET_DIR, 'valid', 'images')
test_img_dir = os.path.join(DATASET_DIR, 'test', 'images')

# สร้างและตั้งค่า data.yaml ให้เป็น Absolute Path ป้องกันปัญหา Path คลาดเคลื่อน
yaml_path = os.path.join(DATASET_DIR, 'data.yaml')
data_config = {
    'train': train_img_dir,
    'val': val_img_dir,
    'nc': 2,
    'names': ['with_helmet', 'without_helmet']
}
if os.path.exists(test_img_dir):
    data_config['test'] = test_img_dir

with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

train_count = len(os.listdir(train_img_dir)) if os.path.exists(train_img_dir) else 0
val_count = len(os.listdir(val_img_dir)) if os.path.exists(val_img_dir) else 0
print("\n📊 สรุปจำนวนรูปภาพในชุดข้อมูล:")
print(f"   - ชุดฝึกสอน (Train): {train_count} รูป")
print(f"   - ชุดประเมินผล (Valid): {val_count} รูป")
print("   - คลาสที่ตรวจจับ: 0 = with_helmet (สวมหมวก), 1 = without_helmet (ไม่สวมหมวก)")

### 4️⃣ ติดตั้ง Ultralytics (YOLO)

In [ ]:
!pip install -q ultralytics
import ultralytics
ultralytics.checks()

### 5️⃣ เริ่มต้นการฝึกสอน (Training) ด้วย YOLOv8 Nano - ปรับแต่งสูตร Pi 4 Max FPS
- **โมเดลตั้งต้น:** `yolov8n.pt` (YOLOv8 Nano มาตรฐาน เสถียรและแม่นยำสูง)
- **ขนาดภาพ (`imgsz=320`):** ฝึกสอนที่ 320x320 พิกเซลโดยตรง เพื่อให้โมเดลจำแนกโครงสร้างขนาดเล็กได้อย่างคมชัดและรันบน Raspberry Pi ได้เร็วที่สุด
- **Batch Size:** 16
- **Epochs:** 70 รอบ (พร้อม Early Stopping 15 รอบ)

In [ ]:
from ultralytics import YOLO

# โหลดโมเดลตั้งต้น YOLOv8 Nano
model_choice = 'yolov8n.pt'
print(f"🚀 กำลังโหลดโมเดลตั้งต้น: {model_choice} (YOLOv8 Nano) ...")
model = YOLO(model_choice)

# เริ่มต้นกระบวนการฝึกสอนที่ขนาด 320x320 โดยตรง
print("\n🔥 เริ่มต้นการฝึกสอนโมเดล YOLOv8n (สูตร 320px Max FPS สำหรับ Raspberry Pi 4)...")
results = model.train(
    data=yaml_path,
    epochs=70,
    patience=15,         # หยุดอัตโนมัติหากค่า Loss ไม่ลดลงใน 15 รอบ
    batch=16,
    imgsz=320,           # ⭐ หัวใจสำคัญ: เทรนที่ 320x320 เพื่อให้โมเดลเร็วที่สุดบน Pi 4
    device=0,
    workers=4,
    project='/content/runs/train',
    name='helmet_yolov8n_pi4',
    save=True,
    plots=True
)
print("🎉 การฝึกสอนโมเดล YOLOv8n เสร็จสมบูรณ์!")

### 6️⃣ แสดงผลกราฟการฝึกสอน (Results & Confusion Matrix)

In [ ]:
from IPython.display import Image, display

run_dir = '/content/runs/train/helmet_yolov8n_pi4'

# 1. แสดงกราฟผลการเทรน (Loss, Precision, Recall, mAP50)
results_plot = os.path.join(run_dir, 'results.png')
if os.path.exists(results_plot):
    print("📈 กราฟการฝึกสอน (Training Curves):")
    display(Image(filename=results_plot, width=850))

# 2. แสดง Confusion Matrix
cm_plot = os.path.join(run_dir, 'confusion_matrix.png')
if os.path.exists(cm_plot):
    print("🎯 ตารางความแม่นยำ (Confusion Matrix):")
    display(Image(filename=cm_plot, width=650))

### 7️⃣ ทดสอบตรวจจับภาพจริงที่ขนาด 320x320

In [ ]:
import glob
import random

best_weights = os.path.join(run_dir, 'weights', 'best.pt')
trained_model = YOLO(best_weights)

# สุ่มภาพจากชุด Valid มา 3 ภาพเพื่อทดสอบทำนายผลที่ความละเอียด 320
val_images = glob.glob(f"{data_config['val']}/*.jpg") + glob.glob(f"{data_config['val']}/*.png")
if val_images:
    sample_images = random.sample(val_images, min(3, len(val_images)))
    trained_model.predict(
        source=sample_images,
        imgsz=320,
        conf=0.35,
        save=True,
        project='/content/runs/predict',
        name='test_samples_pi4',
        exist_ok=True
    )
    
    pred_dir = '/content/runs/predict/test_samples_pi4'
    for img_p in glob.glob(f"{pred_dir}/*.jpg") + glob.glob(f"{pred_dir}/*.png"):
        print(f"🖼️ ผลการตรวจจับ (320px): {os.path.basename(img_p)}")
        display(Image(filename=img_p, width=500))

### 8️⃣ ส่งออกโมเดลฟอร์แมตความเร็วสูง (NCNN + TFLite INT8 + ONNX) สำหรับ Raspberry Pi 4

In [ ]:
# ----------------------------------------------------
# ส่งออกโมเดลด้วยเอนจินประมวลผลความเร็วสูงสำหรับ ARM Cortex-A72
# ----------------------------------------------------
export_files = []

# 1. NCNN Model (Tencent NCNN ARM NEON - เร็วและลื่นไหลที่สุดบน Raspberry Pi 4)
try:
    print("🚀 [1/3] กำลัง Export โมเดล NCNN (imgsz=320)... ")
    ncnn_dir = trained_model.export(format='ncnn', imgsz=320)
    print(f"   ✔️ NCNN สำเร็จ: {ncnn_dir}")
    
    # บีบอัดเป็น best_ncnn_model.zip
    zip_ncnn = '/content/best_ncnn_model'
    shutil.make_archive(zip_ncnn, 'zip', ncnn_dir)
    export_files.append((f"{zip_ncnn}.zip", 'best_ncnn_model.zip'))
    print(f"   📦 บีบอัดไฟล์: {zip_ncnn}.zip")
except Exception as e:
    print(f"   ⚠️ ไม่สามารถ Export NCNN ได้: {e}")

# 2. TFLite INT8 Quantization (ขนาดโมเดลเล็กลงเหลือ ~3 MB และคำนวณแบบ 8-bit Integer บน CPU ARM)
try:
    print("\n⚡ [2/3] กำลัง Export โมเดล TFLite INT8 Quantization (imgsz=320)... ")
    tflite_path = trained_model.export(format='tflite', int8=True, data=yaml_path, imgsz=320)
    print(f"   ✔️ TFLite INT8 สำเร็จ: {tflite_path}")
    export_files.append((tflite_path, 'best_int8_pi4.tflite'))
except Exception as e:
    print(f"   ℹ️ TFLite INT8 ข้ามไป: {e}")

# 3. ONNX Model (Simplified สำหรับรันด้วย ONNXRuntime)
try:
    print("\n📦 [3/3] กำลัง Export โมเดล ONNX (imgsz=320)... ")
    onnx_path = trained_model.export(format='onnx', imgsz=320, simplify=True)
    print(f"   ✔️ ONNX สำเร็จ: {onnx_path}")
    export_files.append((onnx_path, 'best_pi4.onnx'))
except Exception as e:
    print(f"   ℹ️ ONNX ข้ามไป: {e}")

print("\n✅ กระบวนการ Export เสร็จสิ้นเรียบร้อย!")

### 9️⃣ คัดลอกโมเดลและรายงานผลกลับสู่ Google Drive อัตโนมัติ

In [ ]:
# กำหนดโฟลเดอร์ปลายทางใน Google Drive
target_drive_folder = '/content/drive/MyDrive/กันล็อคเด้ออ้าย'
if not os.path.exists(target_drive_folder):
    target_drive_folder = '/content/drive/MyDrive/helmet_model_trained'
os.makedirs(target_drive_folder, exist_ok=True)

# รายการไฟล์หลักที่ต้องบันทึก
all_to_copy = [
    (os.path.join(run_dir, 'weights', 'best.pt'), 'best.pt'),
    (os.path.join(run_dir, 'weights', 'last.pt'), 'last.pt'),
    (os.path.join(run_dir, 'results.png'), 'results.png'),
    (os.path.join(run_dir, 'confusion_matrix.png'), 'confusion_matrix.png')
] + export_files

print(f"💾 กำลังคัดลอกไฟล์ทั้งหมดไปยัง Google Drive: {target_drive_folder} ...\n")
for src, filename in all_to_copy:
    if src and os.path.exists(src):
        dest = os.path.join(target_drive_folder, filename)
        shutil.copy(src, dest)
        size_mb = os.path.getsize(dest) / (1024 * 1024)
        print(f"   ✔️ บันทึกสำเร็จ: {filename} ({size_mb:.2f} MB)")

print("\n🎉 บันทึกทุกไฟล์ลง Google Drive เรียบร้อยแล้ว! ปลอดภัยแม้เซสชัน Colab จะหมดเวลา")

### 🔟 วิธีนำไปรันบน Raspberry Pi 4 ให้ได้ 25 - 40+ FPS:
1. ดาวน์โหลดไฟล์ **`best_ncnn_model.zip`** จาก Google Drive ไปยัง Raspberry Pi
2. แตกไฟล์ด้วยคำสั่ง: `unzip best_ncnn_model.zip`
3. รันโปรแกรมหลัก:
   ```bash
   bash setup_fast_rpi.sh
   # หรือ
   python3 helmet_detection.py
   ```
*(ระบบใน `helmet_detection.py` จะตรวจพบโมเดล NCNN และเปิดใช้งานกล้องแยกเธรด + สลับเฟรมอัจฉริยะให้อัตโนมัติทันที)*